# 1 — Quickstart

**Runs on CPU in well under a minute. No model weights, no dataset download.**

The point of this notebook is not to impress you with a result — it is to get a RECAST panel into
your hands and show you the three objects the API returns, so the next four tutorials have
something to build on.

We use synthetic data and `StubEncoder`, a deterministic stand-in encoder
(`embed = L2_normalize(log1p(X) @ W)`, `W = I`). It is not a foundation model and makes no
biological claims. What it *is* good for is showing the shape of the computation with nothing
hidden: the contrast, the ranking, and the gate are all visible in a toy you can reason about by
hand.

:::{note}
`StubEncoder` lives in `recast.encoders`, which imports torch unconditionally — so even this
CPU-only notebook needs the `[attribution]` install tier, not the core one.
:::

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")  # no ipywidgets in this kernel

import numpy as np, pandas as pd, anndata as ad
import recast
from recast.encoders import StubEncoder

print("recast", recast.__version__)

recast 0.8.0


## A toy where we know the answer

Three populations — `Alpha`, `Beta`, `Gamma` — of 80 cells each, over 39 genes with a deliberate
structure:

- **5 housekeeping genes** (`HK1`–`HK5`), expressed *very* highly in all three populations.
- **3 marker genes per population** (`AMARK1`–`3`, `BMARK…`, `GMARK…`), moderately expressed in
  their own population and near-absent elsewhere.
- **25 noise genes**, low and uninformative everywhere.

The markers are the right answer. The housekeeping genes are the trap: they are by far the
highest-expressed genes in every population, and they say nothing about which population a cell
belongs to.

In [2]:
rng = np.random.default_rng(0)
states = {"Alpha": 80, "Beta": 80, "Gamma": 80}
hk     = [f"HK{i}" for i in range(1, 6)]
marks  = {s: [f"{s[0].upper()}MARK{i}" for i in (1, 2, 3)] for s in states}
noise  = [f"NOISE{i}" for i in range(1, 26)]
genes  = hk + [g for s in states for g in marks[s]] + noise

blocks, labels = [], []
for s, n in states.items():
    B = np.zeros((n, len(genes)))
    for j, g in enumerate(genes):
        if   g in hk:        B[:, j] = rng.poisson(60, n)   # high everywhere -- the trap
        elif g in marks[s]:  B[:, j] = rng.poisson(12, n)   # this population's markers
        elif any(g in v for v in marks.values()):
            B[:, j] = rng.poisson(1, n)                     # another population's markers
        else:                B[:, j] = rng.poisson(4, n)    # noise
    blocks.append(B); labels += [s] * n

adata = ad.AnnData(np.vstack(blocks).astype("float32"))
adata.var_names = genes
adata.obs["state"] = pd.Categorical(labels)
adata

AnnData object with n_obs × n_vars = 240 × 39
    obs: 'state'

## First, what plain expression says

Rank genes for `Alpha` by mean expression — the thing a naive "top expressed genes" readout gives
you.

In [3]:
means = pd.DataFrame(adata.X, columns=genes, index=labels).groupby(level=0).mean()
means.loc["Alpha"].sort_values(ascending=False).head(8).round(2)

HK2       61.490002
HK1       61.259998
HK4       61.209999
HK3       60.340000
HK5       60.290001
AMARK3    12.640000
AMARK1    11.800000
AMARK2    11.680000
Name: Alpha, dtype: float32

Every one of the top hits is a housekeeping gene. This is the failure mode RECAST exists to
avoid, drawn as bluntly as possible: **magnitude is not identity**. A gene can dominate a
population's expression and still carry no information about what makes that population
different from its neighbours.

## Now RECAST

One call. `reference="siblings"` means *every cell that is not the target* — here, the other two
populations. Omitting `target` attributes every population in turn, each against its own
reference.

The call warns, and it is worth reading once: RECAST reads no cell-type hierarchy, so `"siblings"`
and `"rest"` resolve to the same mask. In this toy that mask *is* what we want, because the object
holds exactly the three populations we are comparing. On a real atlas it would not be — asking for
a CD4 memory subtype's "siblings" would hand you B cells and monocytes as the reference. Subset the
object to the lineage first, or pass the sibling labels explicitly.
[Tutorial 3](03_choosing_the_reference) is entirely about this choice.

In [4]:
res = recast.attribute(StubEncoder(len(genes)), adata, "state",
                      reference="siblings", device="cpu")

for s in states:
    print(f"{s:6s} ->", res.top(s, 5))

Alpha  -> ['AMARK1', 'AMARK3', 'AMARK2', 'NOISE19', 'NOISE16']
Beta   -> ['BMARK3', 'BMARK1', 'BMARK2', 'NOISE10', 'NOISE22']
Gamma  -> ['GMARK1', 'GMARK3', 'GMARK2', 'NOISE20', 'NOISE15']


<cell 7>:1: SiblingReferenceWarning: reference='siblings' resolves to every other cell in this AnnData (3 labels present: Alpha, Beta, Gamma). RECAST does not read a cell-type hierarchy, so 'siblings' and 'rest' are the same mask here. If this object is already one lineage, this is the sibling contrast you want; otherwise subset it to the lineage first, or pass the sibling labels explicitly as a list.


All three markers, top-3, for all three populations. The housekeeping genes are gone — not
because they were filtered out, but because they *do not move* along the contrast direction: they
are equally high in the target and in the reference, so they contribute nothing to separating
them.

## The three things you get back

`attribute` returns an `AttributionResult` with three parts worth knowing.

In [5]:
print("1. res.attribution -- raw signed attribution, genes x states")
print("   shape:", res.attribution.shape)
display(res.attribution.head(3).round(4))

print("\n2. res.genes -- every gene, ranked, per state")
print("   lengths:", {s: len(v) for s, v in res.genes.items()}, f"(adata has {adata.n_vars} genes)")

print("\n3. res.top(state, k) -- just the prefix of res.genes")
print("   res.top('Alpha', 3) ==", res.top("Alpha", 3))

1. res.attribution -- raw signed attribution, genes x states
   shape: (39, 3)


,Alpha,Beta,Gamma
HK1,0.0,-0.0,0.0
HK2,0.0,0.0,-0.0
HK3,-0.0,0.0,-0.0



2. res.genes -- every gene, ranked, per state
   lengths: {'Alpha': 39, 'Beta': 39, 'Gamma': 39} (adata has 39 genes)

3. res.top(state, k) -- just the prefix of res.genes
   res.top('Alpha', 3) == ['AMARK1', 'AMARK3', 'AMARK2']


:::{important}
`res.genes[state]` contains **all** of `adata.var_names`, not just the genes that pass the gate.
The ranking keeps a gene ahead of the rest only if it is genuinely **up** in the target relative to
the reference (`dC = centroid(target) − centroid(reference) > 0`, the default `gate="dC"`); genes
with `dC <= 0` are ranked last rather than dropped, because being down in the target is a real
statement — it argues for the *reference* direction. Note the gate reads `dC`, not the sign of the
attribution itself: a gene can pick up a positive attribution and still be down-regulated, and
`gate="dC"` is what keeps that gene out of your panel. "The markers" is always a prefix you choose:
`res.top(state, k)`.
:::

Because we planted the structure, we can group `Alpha`'s 39 genes by what they *are* and look at
the attribution each class receives:

In [6]:
alpha_rank = {g: i for i, g in enumerate(res.genes["Alpha"])}

def gene_class(g):
    if g in marks["Alpha"]:                        return "Alpha's own markers"
    if any(g in v for v in marks.values()):        return "other populations' markers"
    if g in hk:                                    return "housekeeping (high, shared)"
    return "noise"

summary = pd.DataFrame({"attribution": res.attribution["Alpha"],
                        "rank":  [alpha_rank[g] for g in res.attribution.index],
                        "class": [gene_class(g) for g in res.attribution.index]})
(summary.groupby("class")
        .agg(n=("attribution", "size"), mean_attribution=("attribution", "mean"),
             best_rank=("rank", "min"), worst_rank=("rank", "max"))
        .round(4))

,n,mean_attribution,best_rank,worst_rank
class,,,,
Alpha's own markers,3,0.0293,0,2
"housekeeping (high, shared)",5,0.0000,9,31
noise,25,0.0000,3,38
other populations' markers,6,0.0103,18,23


Read that table row by row — it is the whole method in miniature:

- **Alpha's own markers** take ranks 0–2. They have both the highest mean attribution (`+0.029`)
  and, by a wide margin, the highest mean `dC` (`+3.20`): strongly up in Alpha relative to the
  other two populations, and pushing the embedding along the contrast.
- **Housekeeping genes** contribute `0.0000` on average despite being the highest-expressed genes
  in the data. They are equally high in the target and in the reference, so they cancel — the path
  from `C_R` to `C_T` barely moves along those coordinates. This is the row that separates RECAST
  from a "top expressed genes" list.
- **Other populations' markers** have a small *positive* mean attribution (`+0.010`) but a strongly
  negative mean `dC` (`−1.73`): near-absent in Alpha, present in the reference. The `dC` gate is
  what puts them at ranks 18–23 instead of in your panel. This is precisely the sign-mismatch case
  the gate exists for, and why the gate reads `dC` rather than the attribution's own sign.
- **Noise** sits at `0.0000` and spans ranks 3–38.

Two numbers worth keeping apart: 33 of these 39 genes have a positive *attribution*, but only 16
pass the `dC` gate, and those 16 occupy ranks 0–15. Everything below rank 15 is an ordering of
genes the method has already set aside — real values, but not a ranking to read. Take a prefix.

## Contrast QC comes for free

Since v0.5.0 every `attribute` call also returns two embedding-only diagnostics per state, at
negligible cost. `dprime` is how far apart the target and reference sit along the contrast
direction; `cos_u` is how stable that direction is under random half-splits of the cells.

In [7]:
res.qc.round(3)

,n_target,n_reference,dprime,cos_u_mean,cos_u_min
Alpha,80.0,160.0,9.779,0.979,0.968
Beta,80.0,160.0,10.805,0.980,0.968
Gamma,80.0,160.0,10.371,0.979,0.968


`d' ≈ 10` and `cos_u ≈ 0.98` — a clean, well-separated contrast, which is what you would hope for
in a toy with planted markers. [Tutorial 4](04_contrast_qc) shows what these numbers look like
when the contrast is *not* real, and why that case is dangerous enough to deserve its own
notebook.

## Reweighting the ranking

`recast.composite` re-scores the positive-attribution channel by per-gene expression specificity
and discriminativeness computed from the same `adata`. It needs no encoder and no torch.

In [8]:
n_pos = int((res.attribution["Alpha"] > 0).sum())
print(f"genes with positive attribution for Alpha: {n_pos} of {adata.n_vars}")

k = 3   # only compare within the positive channel -- see the note below
pd.DataFrame({m: recast.composite(res, adata, "state", mode=m)["Alpha"][:k]
              for m in ("bare", "tauE", "tauE_discrRU")},
             index=[f"#{i+1}" for i in range(k)])

genes with positive attribution for Alpha: 33 of 39


,bare,tauE,tauE_discrRU
#1,AMARK1,AMARK1,AMARK1
#2,AMARK3,AMARK3,AMARK3
#3,AMARK2,AMARK2,AMARK2


On a toy this clean the modes agree; on real data they do not, and the differences are the point
— see [Composite modes](../usage.md#composite-modes) for what each one rewards.

:::{warning}
We compared only the top 3 on purpose. Only 16 of these 39 genes pass the `dC` gate, and
`composite` ranks gated-out genes last in *every* mode — so beyond rank 16 you would be comparing
the orderings two functions happen to impose on a set of genes that both consider uninformative.
`res.top()` and `composite(mode="bare")` genuinely disagree there, and neither is wrong. Choose `k`
from your data, not from the length of the list.
:::

## Where to go next

- [Tutorial 2](02_subtype_markers) — the same call against a real foundation model and a real
  25,980-cell atlas, plus the two preprocessing contracts that silently ruin results.
- [Tutorial 3](03_choosing_the_reference) — the argument that makes RECAST different from a
  differential-expression test.
- [Tutorial 4](04_contrast_qc) — how to tell a real panel from a confident-looking one.